In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings(
    "ignore", category=DeprecationWarning
)  # to avoid deprecation warnings
# Univarié → Bivarié → Multivarié → Nettoyage → Feature Engineering → Sélection → Split → Entraînement → Évaluatio

In [3]:
df = pd.read_csv("sales_history.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6276060 entries, 0 to 6276059
Data columns (total 8 columns):
 #   Column         Dtype  
---  ------         -----  
 0   sale_date      object 
 1   product_id     int64  
 2   product_name   object 
 3   category_id    int64  
 4   category       object 
 5   qty_sold       int64  
 6   prix_unitaire  float64
 7   prix_total     float64
dtypes: float64(2), int64(3), object(3)
memory usage: 383.1+ MB


In [6]:
print("=" * 50)
print("DIAGNOSTIC DU DATASET")
print("=" * 50)
print(f"Lignes totales      : {len(df):,}")
print(f"Produits distincts  : {df['product_id'].nunique():,}")
print(f"Jours distincts     : {df['sale_date'].nunique():,}")
print(f"Catégories          : {df['category_id'].nunique()}")
print(f"Période             : {df['sale_date'].min()} → {df['sale_date'].max()}")
print(f"Taille mémoire      : {df.memory_usage(deep=True).sum()/1024/1024:.0f} Mo")

# Ventes vs zéros
n_ventes = (df['qty_sold'] > 0).sum()
print(f"\nLignes avec ventes  : {n_ventes:,} ({n_ventes/len(df)*100:.1f}%)")
print(f"Lignes à zéro       : {len(df) - n_ventes:,} ({(len(df)-n_ventes)/len(df)*100:.1f}%)")

# Top 10 produits les plus vendus
print("\nTop 10 produits par volume :")
top = df.groupby(["product_id", "product_name"])["qty_sold"].sum().sort_values(ascending=False).head(10)
print(top.to_string())

# Produits avec très peu de ventes (candidats à filtrer)
ventes_par_produit = df.groupby("product_id")["qty_sold"].sum()
print(f"\nProduits avec 0 vente totale  : {(ventes_par_produit == 0).sum()}")
print(f"Produits avec < 10 ventes     : {(ventes_par_produit < 10).sum()}")
print(f"Produits avec >= 10 ventes    : {(ventes_par_produit >= 10).sum()}")

DIAGNOSTIC DU DATASET
Lignes totales      : 6,276,060
Produits distincts  : 4,102
Jours distincts     : 1,530
Catégories          : 95
Période             : 2022-01-02 → 2026-03-11
Taille mémoire      : 1704 Mo

Lignes avec ventes  : 53,766 (0.9%)
Lignes à zéro       : 6,222,294 (99.1%)

Top 10 produits par volume :
product_id  product_name                          
8055        StarlinK Mini Kit                         5772
8054        StarlinK Standard Kit V4                  5698
14712       Onduleur UPS Technology InLine 850VA      4888
8053        Starlink Standard Actuated Kit            1746
13471       Lenovo B210 Backpack 15.6''                925
14936       Xprinter Imprimante Thermique XP-Q801K     773
13642       JBL Tune 110 Noir                          632
11744       Logitech M171 Noir                         546
14658       Tapis de souris MCL Samar Classic          527
9685        HP 305XL Noir                              525

Produits avec 0 vente totale  : 0
Produi

### EDA (Analysons les données)

In [5]:
# 1. Voir les premières lignes
print("=== Premières lignes ===")
print(df.head())

# 2. Voir les dernières lignes (pour vérifier la cohérence)
print("\n=== Dernières lignes ===")
print(df.tail())


=== Premières lignes ===
    sale_date  product_id                 product_name  category_id  \
0  2022-01-02        7827  Bose QuietComfort QC35 Noir           64   
1  2022-01-03        7827  Bose QuietComfort QC35 Noir           64   
2  2022-01-04        7827  Bose QuietComfort QC35 Noir           64   
3  2022-01-05        7827  Bose QuietComfort QC35 Noir           64   
4  2022-01-06        7827  Bose QuietComfort QC35 Noir           64   

                                  category  qty_sold  prix_unitaire  \
0  All / MARCHANDISES / SON HIFI / Casques         0      1460000.0   
1  All / MARCHANDISES / SON HIFI / Casques         0      1460000.0   
2  All / MARCHANDISES / SON HIFI / Casques         0      1460000.0   
3  All / MARCHANDISES / SON HIFI / Casques         0      1460000.0   
4  All / MARCHANDISES / SON HIFI / Casques         0      1460000.0   

   prix_total  
0         0.0  
1         0.0  
2         0.0  
3         0.0  
4         0.0  

=== Dernières lignes ===

In [6]:
# 3. Connaître la taille du dataset
print(f"\n=== Dimensions ===")
print(f"Lignes : {df.shape[0]}")
print(f"Colonnes : {df.shape[1]}")

# 4. Lister toutes les colonnes
print("\n=== Liste des colonnes ===")
print(df.columns.tolist())

# 5. Obtenir les infos générales
print("\n=== Informations générales ===")
print(df.info())


=== Dimensions ===
Lignes : 6276060
Colonnes : 8

=== Liste des colonnes ===
['sale_date', 'product_id', 'product_name', 'category_id', 'category', 'qty_sold', 'prix_unitaire', 'prix_total']

=== Informations générales ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6276060 entries, 0 to 6276059
Data columns (total 8 columns):
 #   Column         Dtype  
---  ------         -----  
 0   sale_date      object 
 1   product_id     int64  
 2   product_name   object 
 3   category_id    int64  
 4   category       object 
 5   qty_sold       int64  
 6   prix_unitaire  float64
 7   prix_total     float64
dtypes: float64(2), int64(3), object(3)
memory usage: 383.1+ MB
None


### Analysons s'il existe des valeurs manquantes

In [7]:
# Etape 3: Analyse des valeurs manquantes
# Compter les valeurs manquantes
print("=== VALEURS MANQUANTES ===")

# Méthode 1 : Comptage simple
# missing = df.isnull().sum()
# print(missing[missing > 0])  # Afficher seulement les colonnes avec des NaN

# Méthode 2 : Avec pourcentages (PLUS UTILE)
missing_df = pd.DataFrame({
    'Colonne': df.columns,
    'Valeurs_manquantes': df.isnull().sum(),
    'Pourcentage': (df.isnull().sum() / len(df)) * 100
})

missing_df = missing_df[missing_df['Valeurs_manquantes'] > 0].sort_values(
    'Pourcentage', ascending=False
)

if len(missing_df) == 0:
    print("Sans valeur manquantes")

print(missing_df)

=== VALEURS MANQUANTES ===
Sans valeur manquantes
Empty DataFrame
Columns: [Colonne, Valeurs_manquantes, Pourcentage]
Index: []


### Analysons la variable cible s'il y a une desiquilibre (problème de regression)

In [9]:
n_with_sales = (df["qty_sold"] > 0).sum() # nombres des lignes ou il y vente
n_zero = (df["qty_sold"] == 0).sum() # nombre de ligne ou il n'y a pas de vente
pct_sales = n_with_sales / len(df) * 100 # Mettons en pourcentage
pct_zero = n_zero / len(df) * 100 # Metons en pourcentage

In [10]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Répartition des lignes", "Zoom sur les ventes"),
    specs=[[{"type": "bar"}, {"type": "bar"}]],
)
 
# Graphique 1 : Barchart global
fig.add_trace(
    go.Bar(
        x=["Jours sans vente<br>(qty_sold = 0)", "Jours avec vente<br>(qty_sold > 0)"],
        y=[n_zero, n_with_sales],
        text=[f"{pct_zero:.1f}%<br>({n_zero:,})", f"{pct_sales:.1f}%<br>({n_with_sales:,})"],
        textposition="outside",
        marker_color=["#94A3B8", "#F59E0B"],
        textfont=dict(size=14, color=["#475569", "#92400E"]),
    ),
    row=1, col=1,
)
 
# Graphique 2 : Répartition des produits par volume de ventes
ventes_par_produit = df.groupby("product_id")["qty_sold"].sum()
bins = {
    "0 ventes": (ventes_par_produit == 0).sum(),
    "1-9 ventes": ((ventes_par_produit > 0) & (ventes_par_produit < 10)).sum(),
    "10-49 ventes": ((ventes_par_produit >= 10) & (ventes_par_produit < 50)).sum(),
    "50-99 ventes": ((ventes_par_produit >= 50) & (ventes_par_produit < 100)).sum(),
    "100-499": ((ventes_par_produit >= 100) & (ventes_par_produit < 500)).sum(),
    "500+": (ventes_par_produit >= 500).sum(),
}
 
colors = ["#DC2626", "#F59E0B", "#3B82F6", "#10B981", "#8B5CF6", "#06B6D4"]
fig.add_trace(
    go.Bar(
        x=list(bins.keys()),
        y=list(bins.values()),
        text=[str(v) for v in bins.values()],
        textposition="outside",
        marker_color=colors,
        textfont=dict(size=13),
    ),
    row=1, col=2,
)
 
fig.update_layout(
    title=dict(
        text="Dataset avant nettoyage — 99.1% des lignes sont des zéros",
        font=dict(size=18),
    ),
    showlegend=False,
    height=500,
    template="plotly_white",
)
fig.update_yaxes(title_text="Nombre de lignes", row=1, col=1)
fig.update_yaxes(title_text="Nombre de produits", row=1, col=2)
 

fig.show()

Conclusion: le fait que nous avons  introduit des zeros engendrent une désiquilibre au niveau des données, le modele va surement apprendre beaucoup sur ces zeros que la qté_sold.

D'une autre part, il y a au moins une vente pour chaque produit.

Les données dataient de 2022 (5 ans) donc je pense qu'on peut supprimer les lignes dont les ventes sont inferieures à 10. Supprimer aussi les données les lignes contenant starlin

In [ ]:
df[df['product_name'].str.contains('Starlink')] # like an pandas
# categ starlink = All / MARCHANDISES / INFORMATIQUE / RESEAU

,sale_date,product_id,product_name,category_id,category,qty_sold,prix_unitaire,prix_total
162180,2022-01-02,8037,Adaptateur Ethernet Satellite Internet Starlin...,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,351000.0,0.0
162181,2022-01-03,8037,Adaptateur Ethernet Satellite Internet Starlin...,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,351000.0,0.0
162182,2022-01-04,8037,Adaptateur Ethernet Satellite Internet Starlin...,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,351000.0,0.0
162183,2022-01-05,8037,Adaptateur Ethernet Satellite Internet Starlin...,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,351000.0,0.0
162184,2022-01-06,8037,Adaptateur Ethernet Satellite Internet Starlin...,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,351000.0,0.0
...,...,...,...,...,...,...,...,...
5731375,2026-03-07,19217,Câble Starlink V4 45m,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,702000.0,0.0
5731376,2026-03-08,19217,Câble Starlink V4 45m,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,702000.0,0.0
5731377,2026-03-09,19217,Câble Starlink V4 45m,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,702000.0,0.0
5731378,2026-03-10,19217,Câble Starlink V4 45m,102,All / MARCHANDISES / INFORMATIQUE / RESEAU,0,702000.0,0.0


#### Filtrer les jeux de données: ayant au moins 10 ventes et enlever le starlink

In [22]:
# ──────────────────────────────────────────────
# FILTRAGE
# ──────────────────────────────────────────────
print(f"\n{'='*50}")
print("FILTRAGE DU DATASET")
print(f"{'='*50}")

MIN_TOTAL_SALES = 10
EXCLUDE_KEYWORDS = ["starlink", "StarlinK", "Starlink"]
 
# 3a. Filtrer produits avec >= 10 ventes totales
produits_actifs = ventes_par_produit[ventes_par_produit >= MIN_TOTAL_SALES].index
n_before = df["product_id"].nunique()
df_clean = df[df["product_id"].isin(produits_actifs)].copy()
n_after = df_clean["product_id"].nunique()
print(f"\n  Filtre ventes >= {MIN_TOTAL_SALES} :")
print(f"    Avant : {n_before:,} produits")
print(f"    Après : {n_after:,} produits")
print(f"    Retirés : {n_before - n_after:,} produits")
 
# 3b. Retirer Starlink
mask_starlink = df_clean["product_name"].str.contains(
    "|".join(EXCLUDE_KEYWORDS), case=False, na=False
)
n_starlink = df_clean[mask_starlink]["product_id"].nunique()
starlink_products = df_clean[mask_starlink]["product_name"].unique()
df_clean = df_clean[~mask_starlink]
print(f"\n  Filtre Starlink :")
print(f"    Produits retirés : {n_starlink}")
for p in starlink_products:
    print(f"      - {p}")


FILTRAGE DU DATASET

  Filtre ventes >= 10 :
    Avant : 4,102 produits
    Après : 1,426 produits
    Retirés : 2,676 produits

  Filtre Starlink :
    Produits retirés : 15
      - Adaptateur Ethernet Satellite Internet Starlink V2 pour parabole rectangulaire
      - Starlink Standard Actuated Kit
      - StarlinK Standard Kit V4
      - StarlinK Mini Kit
      - Starlink Flat High Performance Kit
      - Starlink Ethernet Adapter
      - Starlink Pole Adapter Gen3
      - Câble Starlink 23m Gen 2
      - Câble Starlink 23m Gen 3
      - Câble Starlink 46m Gen 2
      - Câble Starlink 46m Gen 3
      - Starlink mounting bracket Type 1
      - Starlink mounting bracket Type 2
      - Starlink mounting bracket Type 3
      - Câble Starlink V4 45m


In [23]:
# Statistiques après nettoyage

In [25]:
# 3c. Stats après nettoyage
print(f"\n{'='*50}")
print("DATASET NETTOYÉ")
print(f"{'='*50}")
print(f"  Lignes          : {len(df_clean):,}")
print(f"  Produits        : {df_clean['product_id'].nunique():,}")
print(f"  Catégories      : {df_clean['category_id'].nunique()}")
#print(f"  Période         : {df_clean['sale_date'].min().date()} → {df_clean['sale_date'].max().date()}")
 
n_ventes = (df_clean["qty_sold"] > 0).sum()
print(f"  Lignes ventes   : {n_ventes:,} ({n_ventes/len(df_clean)*100:.1f}%)")
print(f"  Lignes zéros    : {len(df_clean)-n_ventes:,} ({(len(df_clean)-n_ventes)/len(df_clean)*100:.1f}%)")
print(f"  Ventes totales  : {df_clean['qty_sold'].sum():,.0f} unités")


DATASET NETTOYÉ
  Lignes          : 2,158,830
  Produits        : 1,411
  Catégories      : 74
  Lignes ventes   : 44,254 (2.0%)
  Lignes zéros    : 2,114,576 (98.0%)
  Ventes totales  : 66,537 unités


In [26]:
# Sauvegardez dans un fichier
# ──────────────────────────────────────────────
# 4. SAUVEGARDE
# ──────────────────────────────────────────────
import os
DATA_DIR = '.'
output_path = os.path.join(DATA_DIR, "sales_history_clean.csv")
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"\n✓ Dataset nettoyé sauvegardé → {output_path}")
print(f"  Taille : {size_mb:.0f} Mo")


✓ Dataset nettoyé sauvegardé → ./sales_history_clean.csv
  Taille : 244 Mo


Problème avec ce nouveau dataset: il y a toujours une désequilibre autour des données. Le modèle va toujours apprendr sur ce zéro

Mettons en place une fenetre de contexte: on garde les zéros qui sont proches d'une vente (par exemple à ± 7 jours), et on supprime les zéros qui sont isolés, loin de toute vente.

On garde le contexte utile, supprimé le bruit inutile.
Le modèle apprend mieux car il voit un ratio plus équilibré.

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

warnings.filterwarnings(
    "ignore", category=DeprecationWarning
) 

In [25]:
df_clean_step_1 = pd.read_csv("sales_history_clean.csv")

In [26]:
df_clean_step_1.head()

,sale_date,product_id,product_name,category_id,category,qty_sold,prix_unitaire,prix_total
0,2022-01-02,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,0,348000.0,0.0
1,2022-01-03,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,0,348000.0,0.0
2,2022-01-04,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,0,348000.0,0.0
3,2022-01-05,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,0,348000.0,0.0
4,2022-01-06,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,2,348000.0,696000.0


### FEATURES ENGINEERING

#### 1- Features temporelles (saisonnalité)
-- le modele a besoin qu'on décompose la date en morceaux exploitables: day_of_week (0-6) → Lundi=0, Dimanche=6. month (1-12) → Capture la saisonnalité annuelle : "décembre = pic, février = creux".is_weekend, quarter (1-4) 
-- Périodes spéciales (jours fériés, fêtes)
#### 2- Lags (mémoire du passé)
-- Pourquoi ? La meilleure prédiction de demain, c'est souvent ce qui s'est passé récemment. Si un produit s'est vendu 5 fois hier, il y a des chances qu'il se vende encore aujourd'hui.
lag_1(Ventes d'hier. Réaction immédiate), lag_7 , lag_14 ,lag_28
#### 3- Rolling means (tendance lissée)(La moyenne des ventes sur une fenêtre glissante)
-- Pourquoi ? Un lag seul c'est un point isolé — "hier = 0" ne veut pas dire que le produit ne se vend pas. La rolling mean lisse ces variations : "en moyenne cette semaine, ce produit se vend 2.3 par jour".

rolling_mean_7 → Moyenne des 7 derniers jours. Tendance court terme. (Moyenne courte)

rolling_mean_30 → Moyenne du dernier mois. Tendance moyen terme.

rolling_mean_90 → Moyenne des 3 derniers mois. Tendance long terme.

rolling_std_7 → Écart-type sur 7 jours. Si c'est élevé = ventes irrégulières. Si c'est bas = ventes stables. 

#### 4- Tendances (le produit monte ou descend ?) (surtout pour la partie d'obsolescence)
Ratio entre moyennes courtes et longues

-- Pourquoi ? C'est ça qui détecte la croissance vs l'obsolescence. Si la moyenne récente est plus haute que la moyenne ancienne → le produit est en croissance. L'inverse → déclin.

trend_7_30 = rolling_mean_7 / rolling_mean_30

trend_30_90 = rolling_mean_30 / rolling_mean_90

#### 5- Identification produit
-- product_encoded + category_id

-- Pourquoi ? Pour que le modèle sache QUEL produit il prédit. Sans ça, il ne peut pas distinguer un câble USB d'un écran 4K.

-- product_encoded → product_id transformé en entier séquentiel (0, 1, 2, 3...).
category_id → Déjà un entier sur mon CSV. Le modèle apprend que les produits de la même catégorie ont des patterns similaires.


In [27]:
### PRENONS LES DONNEES DEPUIS 2023
print(f"\n Dataset chargé : {len(pd.read_csv("sales_history_clean.csv")):,} lignes")

df_clean_step_1["sale_date"] = pd.to_datetime(df_clean_step_1["sale_date"])

df_clean_step_1 = df_clean_step_1[df_clean_step_1["sale_date"] >= "2023-01-01"].copy()

print(f" Filtre >= 2023 : {len(df_clean_step_1):,} lignes")

print(f"  Produits : {df_clean_step_1['product_id'].nunique():,}")

n_ventes = (df_clean_step_1["qty_sold"] > 0).sum()

print(f"  Ventes > 0 : {n_ventes:,} ({n_ventes/len(df_clean_step_1)*100:.1f}%)")


 Dataset chargé : 2,158,830 lignes
 Filtre >= 2023 : 1,645,226 lignes
  Produits : 1,411
  Ventes > 0 : 34,566 (2.1%)


### CRÉATION DES FEATURES TEMPORELLES

In [28]:
print("\n  Création des features...")

# on va convertir la date d'abord
 
# 2a. Features de base depuis la date
df_clean_step_1["day_of_week"] = df_clean_step_1["sale_date"].dt.dayofweek       # 0=Lundi, 6=Dimanche
df_clean_step_1["day_of_month"] = df_clean_step_1["sale_date"].dt.day             # 1-31
df_clean_step_1["week_of_year"] = df_clean_step_1["sale_date"].dt.isocalendar().week.astype(int)  # 1-52
df_clean_step_1["month"] = df_clean_step_1["sale_date"].dt.month                  # 1-12
df_clean_step_1["quarter"] = df_clean_step_1["sale_date"].dt.quarter              # 1-4
df_clean_step_1["year"] = df_clean_step_1["sale_date"].dt.year
df_clean_step_1["is_weekend"] = (df_clean_step_1["day_of_week"] >= 5).astype(int) # 0 ou 1
df_clean_step_1["is_month_start"] = (df_clean_step_1["day_of_month"] <= 5).astype(int)
df_clean_step_1["is_month_end"] = (df_clean_step_1["day_of_month"] >= 25).astype(int)
 
print("  ✓ day_of_week, day_of_month, week_of_year, month, quarter, year")
print("  ✓ is_weekend, is_month_start, is_month_end")
 
# 2b. Jours fériés (adapter selon ton pays — ici Madagascar + universels)
fixed_holidays = [
    (1, 1),    # Nouvel An
    (3, 8),    # Journée de la femme
    (3, 29),   # Jour des martyrs
    (5, 1),    # Fête du travail
    (6, 26),   # Fête nationale Madagascar
    (8, 15),   # Assomption
    (11, 1),   # Toussaint
    (12, 25),  # Noël
]
 
df_clean_step_1["is_holiday"] = 0
for m, d in fixed_holidays:
    mask = (df_clean_step_1["sale_date"].dt.month == m) & (df_clean_step_1["sale_date"].dt.day == d)
    df_clean_step_1.loc[mask, "is_holiday"] = 1
 
# Veille de jour férié
holiday_dates = set(df_clean_step_1.loc[df_clean_step_1["is_holiday"] == 1, "sale_date"].dt.date)
df_clean_step_1["is_pre_holiday"] = df_clean_step_1["sale_date"].apply(
    lambda x: 1 if (x.date() + timedelta(days=1)) in holiday_dates else 0
)
 
print("   is_holiday, is_pre_holiday")
 
# 2c. Périodes spéciales
df_clean_step_1["is_christmas_period"] = (
    (df_clean_step_1["sale_date"].dt.month == 12) & (df_clean_step_1["sale_date"].dt.day >= 15)
).astype(int)
 
df_clean_step_1["is_summer"] = df_clean_step_1["sale_date"].dt.month.isin([6, 7, 8]).astype(int)
 
df_clean_step_1["is_back_to_school"] = (
    (df_clean_step_1["sale_date"].dt.month == 9) & (df_clean_step_1["sale_date"].dt.day <= 15)
).astype(int)
 
df_clean_step_1["is_black_friday_period"] = (
    (df_clean_step_1["sale_date"].dt.month == 11) & (df_clean_step_1["sale_date"].dt.day >= 20)
).astype(int)
 
print("   is_christmas_period, is_summer, is_back_to_school, is_black_friday_period")


  Création des features...


  ✓ day_of_week, day_of_month, week_of_year, month, quarter, year
  ✓ is_weekend, is_month_start, is_month_end
   is_holiday, is_pre_holiday
   is_christmas_period, is_summer, is_back_to_school, is_black_friday_period


In [29]:
df_clean_step_1.head()

,sale_date,product_id,product_name,category_id,category,qty_sold,prix_unitaire,prix_total,day_of_week,day_of_month,...,year,is_weekend,is_month_start,is_month_end,is_holiday,is_pre_holiday,is_christmas_period,is_summer,is_back_to_school,is_black_friday_period
364,2023-01-01,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,0,373000.0,0.0,6,1,...,2023,1,1,0,1,0,0,0,0,0
365,2023-01-02,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,0,373000.0,0.0,0,2,...,2023,0,1,0,0,0,0,0,0,0
366,2023-01-03,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,0,373000.0,0.0,1,3,...,2023,0,1,0,0,0,0,0,0,0
367,2023-01-04,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,0,373000.0,0.0,2,4,...,2023,0,1,0,0,0,0,0,0,0
368,2023-01-05,7839,The Legend of Zelda Breath of the Wild Nintend...,128,All / MARCHANDISES / GAMING / Gaming Jeux Vidéos,1,373000.0,373000.0,3,5,...,2023,0,1,0,0,0,0,0,0,0


#### FAISONS D'ABORD UN EDA PAR RAPPORT A CE CHANGEMENT

In [32]:
print("\n" + "-" * 40)
print("EDA — ANALYSE DE LA SAISONNALITÉ")
print("-" * 40)
 
# Ne prendre que les lignes avec ventes pour l'analyse
ventes = df_clean_step_1[df_clean_step_1["qty_sold"] > 0]

# 3a. Ventes par jour de la semaine
print("\n  Ventes moyennes par jour de la semaine :")
jours_noms = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
ventes_par_jour = ventes.groupby("day_of_week")["qty_sold"].sum()
for dow, total in ventes_par_jour.items():
    bar = "█" * int(total / ventes_par_jour.max() * 30)
    print(f"    {jours_noms[dow]:10s} {total:>8,.0f}  {bar}")
 
# 3b. Ventes par mois
print("\n  Ventes totales par mois :")
mois_noms = ["Jan", "Fév", "Mar", "Avr", "Mai", "Jun", "Jul", "Aoû", "Sep", "Oct", "Nov", "Déc"]
ventes_par_mois = ventes.groupby("month")["qty_sold"].sum()
for m, total in ventes_par_mois.items():
    bar = "█" * int(total / ventes_par_mois.max() * 30)
    print(f"    {mois_noms[m-1]:4s} {total:>8,.0f}  {bar}")
 
# 3c. Weekend vs semaine
ventes_semaine = ventes[ventes["is_weekend"] == 0]["qty_sold"].sum()
ventes_weekend = ventes[ventes["is_weekend"] == 1]["qty_sold"].sum()
jours_semaine = df_clean_step_1[df_clean_step_1["is_weekend"] == 0]["sale_date"].nunique()
jours_weekend = df_clean_step_1[df_clean_step_1["is_weekend"] == 1]["sale_date"].nunique()
print(f"\n  Semaine vs Weekend :")
print(f"    Semaine : {ventes_semaine:,.0f} ventes / {jours_semaine} jours = {ventes_semaine/jours_semaine:.1f}/jour")
print(f"    Weekend : {ventes_weekend:,.0f} ventes / {jours_weekend} jours = {ventes_weekend/jours_weekend:.1f}/jour")
 
# 3d. Périodes spéciales
print(f"\n  Impact des périodes spéciales (ventes moyennes/jour) :")
for period_col, period_name in [
    ("is_christmas_period", "Noël (15-31 déc)"),
    ("is_black_friday_period", "Black Friday (20-30 nov)"),
    ("is_summer", "Été (jun-jul-aoû)"),
    ("is_back_to_school", "Rentrée (1-15 sep)"),
    ("is_holiday", "Jours fériés"),
    ("is_pre_holiday", "Veilles de fériés"),
]:
    avg_in = ventes[ventes[period_col] == 1]["qty_sold"].mean() if (ventes[period_col] == 1).any() else 0
    avg_out = ventes[ventes[period_col] == 0]["qty_sold"].mean()
    diff = ((avg_in / avg_out - 1) * 100) if avg_out > 0 else 0
    indicator = "↗" if diff > 5 else ("↘" if diff < -5 else "→")
    print(f"    {period_name:25s} {indicator} {diff:+.1f}% vs reste")


----------------------------------------
EDA — ANALYSE DE LA SAISONNALITÉ
----------------------------------------

  Ventes moyennes par jour de la semaine :
    Lundi         7,060  █████████████████████
    Mardi         8,862  ███████████████████████████
    Mercredi      7,310  ██████████████████████
    Jeudi         8,063  █████████████████████████
    Vendredi      8,024  ████████████████████████
    Samedi        9,642  ██████████████████████████████
    Dimanche      2,621  ████████

  Ventes totales par mois :
    Jan     4,901  ████████████████████
    Fév     4,011  ████████████████
    Mar     3,969  ████████████████
    Avr     3,566  ██████████████
    Mai     3,483  ██████████████
    Jun     3,721  ███████████████
    Jul     3,880  ████████████████
    Aoû     3,763  ███████████████
    Sep     4,250  █████████████████
    Oct     4,297  █████████████████
    Nov     4,515  ██████████████████
    Déc     7,226  ██████████████████████████████

  Semaine vs Weekend :


### AFFICHE SUR UN GRAPHIQUE

In [35]:
print("\n  Génération des graphiques...")
 
# Graphique 1 : Ventes par jour de la semaine + par mois
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Ventes par jour de la semaine",
        "Ventes par mois",
        "Ventes par trimestre",
        "Tendance mensuelle dans le temps",
    ),
    vertical_spacing=0.12,
)
 
# Jour de la semaine
fig.add_trace(
    go.Bar(
        x=jours_noms,
        y=ventes_par_jour.values,
        marker_color=["#3B82F6"]*5 + ["#F59E0B"]*2,
        text=[f"{v:,.0f}" for v in ventes_par_jour.values],
        textposition="outside",
    ),
    row=1, col=1,
)
 
# Par mois
fig.add_trace(
    go.Bar(
        x=mois_noms,
        y=ventes_par_mois.values,
        marker_color="#10B981",
        text=[f"{v:,.0f}" for v in ventes_par_mois.values],
        textposition="outside",
    ),
    row=1, col=2,
)
 
# Par trimestre
ventes_par_trimestre = ventes.groupby("quarter")["qty_sold"].sum()
fig.add_trace(
    go.Bar(
        x=["Q1", "Q2", "Q3", "Q4"],
        y=ventes_par_trimestre.values,
        marker_color="#8B5CF6",
        text=[f"{v:,.0f}" for v in ventes_par_trimestre.values],
        textposition="outside",
    ),
    row=2, col=1,
)
 
# Tendance mensuelle
monthly_trend = (
    ventes.groupby([ventes["sale_date"].dt.to_period("M")])["qty_sold"]
    .sum()
    .reset_index()
)
monthly_trend["sale_date"] = monthly_trend["sale_date"].astype(str)
fig.add_trace(
    go.Scatter(
        x=monthly_trend["sale_date"],
        y=monthly_trend["qty_sold"],
        mode="lines+markers",
        line=dict(color="#DC2626", width=2),
        marker=dict(size=5),
    ),
    row=2, col=2,
)
 
fig.update_layout(
    height=700,
    showlegend=False,
    title="EDA — Saisonnalité des ventes",
    template="plotly_white",
) 

fig.show()


  Génération des graphiques...



### SAUVEGARDE

In [38]:
import os
DATA_DIR = '.'
output_path = os.path.join(DATA_DIR, "sales_step1_temporal.csv")
df_clean_step_1.to_csv(output_path, index=False, encoding="utf-8-sig")

In [40]:
print(f"\n{'='*60}")
print("RÉSUMÉ ÉTAPE 1")
print(f"{'='*60}")
print(f"  Features créées : 15")
print(f"  Colonnes totales : {len(df_clean_step_1.columns)}")
print(f"  Liste : {list(df_clean_step_1.columns)}")

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"  Fichier sauvegardé : {output_path} ({size_mb:.0f} Mo)")




RÉSUMÉ ÉTAPE 1
  Features créées : 15
  Colonnes totales : 23
  Liste : ['sale_date', 'product_id', 'product_name', 'category_id', 'category', 'qty_sold', 'prix_unitaire', 'prix_total', 'day_of_week', 'day_of_month', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend', 'is_month_start', 'is_month_end', 'is_holiday', 'is_pre_holiday', 'is_christmas_period', 'is_summer', 'is_back_to_school', 'is_black_friday_period']
  Fichier sauvegardé : ./sales_step1_temporal.csv (241 Mo)


### ETAPE 2: LAGS

"""
Smart Reassort — Feature Engineering Étape 2
==============================================
Features de Lag (mémoire du passé)
 
Pour chaque produit, on regarde les ventes des jours précédents :
  - lag_1  : ventes d'hier
  - lag_3  : ventes il y a 3 jours
  - lag_7  : ventes il y a 1 semaine
  - lag_14 : ventes il y a 2 semaines
  - lag_21 : ventes il y a 3 semaines
  - lag_28 : ventes il y a 4 semaines
 
Usage : python step2_lag_features.py
"""

In [41]:
import pandas as pd
import numpy as np
import os
 
DATA_DIR = "."

In [42]:
df_2 = pd.read_csv("sales_step1_temporal.csv",encoding="utf-8-sig")
df_2["sale_date"] = pd.to_datetime(df_2["sale_date"])
print(f"\n✓ Dataset chargé : {len(df_2):,} lignes, {len(df_2.columns)} colonnes")


✓ Dataset chargé : 1,645,226 lignes, 23 colonnes


In [ ]:
## CRÉATION DES LAGS

In [44]:
print("\n  Tri par produit + date...")
df_2 = df_2.sort_values(["product_id", "sale_date"]).reset_index(drop=True)
 
lags = [1, 3, 7, 14, 21, 28]
 
print("  Création des lags (par produit)...")
for lag in lags:
    col_name = f"lag_{lag}"
    df_2[col_name] = df_2.groupby("product_id")["qty_sold"].shift(lag)
    n_nan = df_2[col_name].isna().sum()
    print(f"     {col_name:8s} — NaN : {n_nan:,} ({n_nan/len(df_2)*100:.1f}%)")
 



  Tri par produit + date...
  Création des lags (par produit)...
     lag_1    — NaN : 1,411 (0.1%)
     lag_3    — NaN : 4,233 (0.3%)
     lag_7    — NaN : 9,877 (0.6%)
     lag_14   — NaN : 19,754 (1.2%)
     lag_21   — NaN : 29,631 (1.8%)
     lag_28   — NaN : 39,508 (2.4%)


### VÉRIFICATION — EXEMPLE CONCRET

In [45]:
print("\n" + "-" * 40)
print("VÉRIFICATION — Exemple concret")
print("-" * 40)
 
# Trouver un produit avec des ventes pour montrer l'exemple
ventes_par_produit = df_2.groupby("product_id")["qty_sold"].sum().sort_values(ascending=False)
top_product_id = ventes_par_produit.index[0]
top_product_name = df_2[df_2["product_id"] == top_product_id]["product_name"].iloc[0]
 
print(f"\n  Produit : {top_product_name} (id={top_product_id})")
 
# Prendre un extrait avec des ventes
product_data = df_2[df_2["product_id"] == top_product_id].copy()
ventes_idx = product_data[product_data["qty_sold"] > 0].index
 
if len(ventes_idx) > 0:
    # Prendre une fenêtre autour d'une vente
    center_idx = ventes_idx[len(ventes_idx) // 2]  # Milieu de l'historique
    pos = product_data.index.get_loc(center_idx)
    start = max(0, pos - 3)
    end = min(len(product_data), pos + 4)
    sample = product_data.iloc[start:end]
 
    cols_to_show = ["sale_date", "qty_sold"] + [f"lag_{l}" for l in lags]
    print(f"\n{sample[cols_to_show].to_string(index=False)}")
 
    print(f"\n  Lecture : lag_7 = la valeur de qty_sold 7 lignes plus haut")
    print(f"  (= ce même produit, 7 jours avant)")


----------------------------------------
VÉRIFICATION — Exemple concret
----------------------------------------

  Produit : Onduleur UPS Technology InLine 850VA (id=14712)

 sale_date  qty_sold  lag_1  lag_3  lag_7  lag_14  lag_21  lag_28
2024-08-27         3    1.0    3.0    4.0     2.0     1.0     2.0
2024-08-28         2    3.0    1.0    0.0     0.0     1.0     1.0
2024-08-29         2    2.0    1.0    4.0     1.0     2.0     1.0
2024-08-30         7    2.0    3.0    1.0     5.0     1.0     5.0
2024-08-31         5    7.0    2.0    3.0     1.0     0.0     1.0
2024-09-01         0    5.0    2.0    1.0     0.0     1.0     0.0
2024-09-02         2    0.0    7.0    1.0     1.0     0.0     0.0

  Lecture : lag_7 = la valeur de qty_sold 7 lignes plus haut
  (= ce même produit, 7 jours avant)


In [46]:
# ──────────────────────────────────────────────
# 4. STATS SUR LES LAGS
# ──────────────────────────────────────────────
print("\n" + "-" * 40)
print("CORRÉLATION LAGS → QTY_SOLD")
print("-" * 40)
 
# Corrélation entre chaque lag et qty_sold (sur les lignes complètes)
print("\n  Plus la corrélation est haute, plus le lag est prédictif :")
df_complete = df_2.dropna(subset=[f"lag_{l}" for l in lags])
for lag in lags:
    corr = df_complete["qty_sold"].corr(df_complete[f"lag_{lag}"])
    bar = "█" * int(abs(corr) * 50)
    print(f"    lag_{lag:2d} ↔ qty_sold : {corr:.4f}  {bar}")


----------------------------------------
CORRÉLATION LAGS → QTY_SOLD
----------------------------------------

  Plus la corrélation est haute, plus le lag est prédictif :
    lag_ 1 ↔ qty_sold : 0.0443  ██
    lag_ 3 ↔ qty_sold : 0.0404  ██
    lag_ 7 ↔ qty_sold : 0.0453  ██
    lag_14 ↔ qty_sold : 0.0413  ██
    lag_21 ↔ qty_sold : 0.0445  ██
    lag_28 ↔ qty_sold : 0.0390  █


In [48]:
output_path = os.path.join(DATA_DIR, "sales_step2_lags.csv")
df_2.to_csv(output_path, index=False, encoding="utf-8-sig")
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"\n{'='*60}")
print("RÉSUMÉ ÉTAPE 2")
print(f"{'='*60}")
print(f"  Nouvelles features : {len(lags)} lags ({', '.join([f'lag_{l}' for l in lags])})")
print(f"  Colonnes totales   : {len(df_2.columns)}")
print(f"  Fichier sauvegardé : {output_path} ({size_mb:.0f} Mo)")


RÉSUMÉ ÉTAPE 2
  Nouvelles features : 6 lags (lag_1, lag_3, lag_7, lag_14, lag_21, lag_28)
  Colonnes totales   : 29
  Fichier sauvegardé : ./sales_step2_lags.csv (278 Mo)
